# THIS CODE IS BUILT TO RUN ON GOOGLE COLAB ONLY

Setup Steps (Pre-Run):

1) Click on the Files icon to the right on the bottom. By default, you should be in the /content folder, but if not, then navigate to /content from the root (right-click on /content and choose "Open").

2) Upload the ROM included to the /content folder.

3) It is recommended to "Disconnect and delete runtime" (under Runtime menu) before running just to ensure that old data from previous runtimes will not be included.

On first run, the /video folder will be created under /content, and a video of the frames will be placed there with a .json file.  These will be replaced each time the program is run, so save a copy of the video after each run (the video must be recorded each time because Colab does not have a direct display capability).  The .json file does not serve a purpose for this project.

In [ ]:
# Installs and Setup
!apt-get update
!apt-get install -y libglu1-mesa-dev freeglut3-dev mesa-common-dev

!pip install opencv-python
!pip install optuna
!pip install stable-baselines3[extra]
!pip install stable-retro gymnasium
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

LOG_DIR = './logs/'   # Logging directory
OPT_DIR = './opt/'    # Optimized model directory
CHK_DIR = './train/'  # Training directory

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,665 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [77.5 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [8,932 kB]
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,722 kB]
Get:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease [24.3 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 Packages [34.2 kB]
Get:12 http://security.ubuntu.com/ubuntu jamm

In [ ]:
# Imports

import cv2
import gymnasium as gym
import numpy as np
import optuna
import os
import pyglet
import retro
import retro.data
import subprocess
import time
import warnings

from gymnasium import Env
from gymnasium.spaces import Box, MultiBinary
from gymnasium.wrappers import RecordVideo

from IPython import get_ipython

from matplotlib import pyplot as plt

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack

In [ ]:
# Load the Rom - retro.data.list_games() # to see supported

!python -m retro.import .

Importing MortalKombat-Genesis
Imported 1 games


In [ ]:
# Objective Function

def objective(trial):

    return {

        'n_steps': trial.suggest_int('n_steps', 2048, 8192),
        'gamma': trial.suggest_loguniform('gamma', 0.8, 0.9999),
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-5, 1e-4),
        'clip_range': trial.suggest_uniform('clip_range', 0.1, 0.4),
        'gae_lambda': trial.suggest_uniform('gae_lambda', 0.8, 0.99),

    }

In [ ]:
# Optimize Function

def optimize(trial):

    try:

        model_params = objective(trial)

        env = MortalKombat()
        env = Monitor(env, LOG_DIR)
        env = DummyVecEnv([lambda: env])
        env = VecFrameStack(env, 4, channels_order='last')

        model = PPO('CnnPolicy', env, verbose = 0, tensorboard_log = LOG_DIR, **model_params)
        model.learn(total_timesteps = 100) #big 0's = bigger trial

        mean_reward, _ = evaluate_policy(model, env, n_eval_episodes = 1)
        env.close()

        SAVE_PATH = os.path.join(OPT_DIR, 'trial_{}_best_model'.format(trial.number))
        model.save(SAVE_PATH)

        return mean_reward

    except Exception as e:

        return -1


In [ ]:
# Main class

class MortalKombat(gym.Env):

    def __init__(self, render_mode = 'rgb_array'):

        super().__init__()
        self.observation_space = Box(low = 0, high = 255, shape = (84, 84, 1), dtype = np.uint8)
        self.action_space = MultiBinary(12)
        self.game = retro.make(game = 'MortalKombat-Genesis', use_restricted_actions = retro.Actions.FILTERED, render_mode='rgb_array')
        self.previous_frame = None
        self.score = 0
        self.frame_counter = 0
        self.game_over = False
        self.game_start = False

    def step(self, action):

        obs, reward, terminated, _, info = self.game.step(action)
        obs = self.preprocess(obs)
        frame_delta = obs - self.previous_frame
        self.previous_frame = obs
        reward = info['score'] - self.score
        self.score = info['score']
        truncated = False
        done = terminated or self.game_over
        info = info
        self.frame_counter += 1
        return frame_delta, reward, terminated, truncated, info

    def render(self, *args, **kwargs):

        self.game.render()

    def reset(self, **kwargs):

        _ = kwargs.get("seed", None)
        obs, info = self.game.reset()
        obs = self.preprocess(obs)
        self.previous_frame = obs
        self.score = 0
        return obs, info

    def close(self):

        self.game.close()

    def preprocess(self, observation):

        gray = cv2.cvtColor(observation, cv2.COLOR_BGR2GRAY)
        resize = cv2.resize(gray, (84, 84), interpolation = cv2.INTER_CUBIC)
        channels  = np.reshape(resize, (84, 84, 1))
        return channels


In [ ]:
# Callback Function

class TrainAndLoggingCallback(BaseCallback):

    def __init__(self, check_freq, save_path, verbose = 1):

        super(TrainAndLoggingCallback, self).__init__(verbose)
        self.check_freq = check_freq
        self.save_path = save_path

    def _init_callback(self):

        if self.save_path is not None:
            os.makedirs(self.save_path, exist_ok = True)

    def _on_step(self):

        if self.n_calls % self.check_freq == 0:
            model_path = os.path.join(self.save_path, 'best_model_{}'.format(self.n_calls))
            self.model.save(model_path)

        return True

In [ ]:
# Main TRAINING Block

try:

    env.close()

except NameError:

    pass

# Create study based on optimize function
# This is to create the trial files that will prime the later run
study = optuna.create_study(direction = 'maximize')
study.optimize(optimize, n_trials = 5, n_jobs = 1)
study.best_params
study.best_trial.number

# Set up callback - check_freq = count at which every data is recorded
callback = TrainAndLoggingCallback(check_freq = 10000, save_path = CHK_DIR)

# Train model
env = MortalKombat()
env = Monitor(env, LOG_DIR)
env = DummyVecEnv([lambda: env])
env = VecFrameStack(env, 4, channels_order='last')

# Take the best params from the study
model_params = study.best_params

# Apply the PPO policy to the model using the best params
model = PPO('CnnPolicy', env, verbose = 1, tensorboard_log = LOG_DIR, **model_params)

[I 2025-05-13 23:19:52,318] A new study created in memory with name: no-name-927ef42d-5a4d-4b54-aecb-f1b4479f22ae
<ipython-input-4-da9cab68c047>:8: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'gamma': trial.suggest_loguniform('gamma', 0.8, 0.9999),
<ipython-input-4-da9cab68c047>:9: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 1e-5, 1e-4),
<ipython-input-4-da9cab68c047>:10: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  'clip_range': trial.suggest_uniform('clip_range', 0

Using cpu device
Wrapping the env in a VecTransposeImage.


/usr/local/lib/python3.11/dist-packages/stable_baselines3/ppo/ppo.py:155: UserWarning: You have specified a mini-batch size of 64, but because the `RolloutBuffer` is of size `n_steps * n_envs = 4666`, after every 72 untruncated mini-batches, there will be a truncated mini-batch of size 58
We recommend using a `batch_size` that is a factor of `n_steps * n_envs`.
Info: (n_steps=4666 and n_envs=1)
  warnings.warn(


In [ ]:
# the best trial model - hyperparameter tuned
model.load(os.path.join(OPT_DIR, 'trial_{}_best_model'.format(study.best_trial.number)))

In [ ]:
# Now we let the model learn
# Increase total_timesteps by order of 10k to generate more tests to learn from
model.learn(total_timesteps = 10000, callback = callback) #extra 0's = bigger trial iterations

Logging to ./logs/PPO_6
-----------------------------
| time/              |      |
|    fps             | 197  |
|    iterations      | 1    |
|    time_elapsed    | 23   |
|    total_timesteps | 4666 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 67          |
|    iterations           | 2           |
|    time_elapsed         | 139         |
|    total_timesteps      | 9332        |
| train/                  |             |
|    approx_kl            | 0.009673437 |
|    clip_fraction        | 0.0926      |
|    clip_range           | 0.221       |
|    entropy_loss         | -8.31       |
|    explained_variance   | 1.31e-06    |
|    learning_rate        | 2.53e-05    |
|    loss                 | 2.38e+05    |
|    n_updates            | 10          |
|    policy_gradient_loss | 0.000472    |
|    value_loss           | 9.29e+06    |
-----------------------------------------
----------

In [ ]:
# Take the best model created from the output
model = PPO.load('./train/best_model_10000.zip')

In [ ]:
# Evaluate the mean reward
mean_reward, _ = evaluate_policy(model, env, n_eval_episodes = 5, render = False)

In [ ]:
# Check the mean reward here - is it > 0 ?
mean_reward

0.0

In [ ]:
# With Training
try:

    env.close()

except NameError:

    pass

env = MortalKombat()
video_every = 100000
env.render_mode = 'rgb_array'
env = RecordVideo(env, "./video", episode_trigger=lambda episode_id: (episode_id % video_every) == 0)
env = DummyVecEnv([lambda: env])
env = VecFrameStack(env, 4, channels_order='last')

obs = env.reset()
done = False
for game in range(1):
    while not done:
        if done:
            obs = env.reset()
        env.venv.envs[0].render()
        action, _ = model.predict(obs[0], deterministic=True)
        action = np.array(env.action_space.sample()).astype(int)
        obs, reward, terminated, info = env.step([action])
        truncated = info[0].get('TimeLimit.truncated', False)
        done = terminated or truncated
        if reward > 0:
            print(reward)
env.close()
info

/usr/local/lib/python3.11/dist-packages/gymnasium/wrappers/rendering.py:395: UserWarning: WARN: Ignored saving a video as there were zero frames to save.
  logger.warn("Ignored saving a video as there were zero frames to save.")
/usr/local/lib/python3.11/dist-packages/gymnasium/wrappers/rendering.py:323: UserWarning: WARN: Recording stopped: expected type of frame returned by render to be a numpy array, got instead <class 'NoneType'>.
  logger.warn(


[1000.]
[500.]
[2000.]
[500.]
[2000.]
[2000.]
[500.]
[2000.]
[50000.]
[122000.]
[5000.]
[5000.]
[1000.]
[2000.]
[500.]
[500.]
[500.]
[500.]
[5000.]
[50000.]
[123000.]
[2000.]
[2000.]
[2000.]
[500.]
[500.]
[500.]
[500.]
[2000.]
[2000.]
[1000.]
[5000.]
[50000.]
[91500.]
[5000.]
[2000.]
[2000.]
[1000.]
[2000.]
[2000.]
[500.]
[2000.]
[50000.]
[113500.]
[2000.]
[500.]
[5000.]


[{'enemy_matches_won': 2,
  'score': 722500,
  'TimeLimit.truncated': False,
  'terminal_observation': array([[[  0,   0,   0,   0],
          [  0,   0,   0,   0],
          [  0,   0,   0,   0],
          ...,
          [  0,   0,   0,   0],
          [  0,   0,   0,   0],
          [  0,   0,   0,   0]],
  
         [[  0,   0,   0,   0],
          [  0,   0,   0,   0],
          [  0,   0,   0,   0],
          ...,
          [  0,   0,   0,   0],
          [  0,   0,   0,   0],
          [  0,   0,   0,   0]],
  
         [[  0,   0,   0,   0],
          [  0,   0,   0,   0],
          [  0,   0,   0,   0],
          ...,
          [  0,   0,   0,   0],
          [  0,   0,   0,   0],
          [  0,   0,   0,   0]],
  
         ...,
  
         [[241,   0, 254,  24],
          [222,   0,  34, 225],
          [222,   0,  33,   1],
          ...,
          [ 32,   0, 223,   0],
          [ 34,   0, 215,  12],
          [252,   0, 241, 248]],
  
         [[  0,   0,   0,   0],
      

In [ ]:
# Without Training

try:

    env.close()

except NameError:

    pass

env = retro.make(game='MortalKombat-Genesis', render_mode='rgb_array')
video_every = 100000
env = RecordVideo(env, "./video", episode_trigger=lambda episode_id: (episode_id % video_every) == 0)

obs = env.reset()
done = False
for game in range(1):
    while not done:
        if done:
            obs = env.reset()
        env.render()
        obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
        done = terminated or truncated
        if reward > 0:
            print(reward)
env.close()
info

/usr/local/lib/python3.11/dist-packages/gymnasium/wrappers/rendering.py:283: UserWarning: WARN: Overwriting existing videos at /content/video folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


500.0
500.0
2000.0
500.0
5000.0
2000.0
2000.0
500.0
5000.0
50000.0
97500.0
5000.0
500.0
1000.0
2000.0
2000.0
2000.0
5000.0
2000.0
500.0
50000.0
114500.0
2000.0
5000.0
2000.0
500.0
2000.0
1000.0
2000.0
50000.0
97000.0
500.0
500.0
5000.0
500.0
2000.0
5000.0
2000.0
2000.0
2000.0
5000.0


/usr/local/lib/python3.11/dist-packages/moviepy/config_defaults.py:1: DeprecationWarning: invalid escape sequence '\P'
  """


{'enemy_matches_won': 2, 'score': 536000}